In [ ]:
# ==========================================
# 1. TUNED ArcFace Loss (For 2D Stability)
# ==========================================
# We lower the scale (s) from 30.0 to 15.0 and margin (m) from 0.5 to 0.25.
# This stops gradients from exploding in a tiny 2D space.
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=15.0, m=0.25):
        super(ArcFaceLoss, self).__init__()
        self.s = s
        self.m = m
        self.W = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.W)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, labels):
        W_norm = F.normalize(self.W, p=2, dim=1)
        x_norm = F.normalize(x, p=2, dim=1)

        cos_theta = F.linear(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_cos = cos_theta[torch.arange(x.size(0)), labels]

        sine = torch.sqrt((1.0 - target_cos**2).clamp(min=1e-7))
        cos_theta_m = target_cos * self.cos_m - sine * self.sin_m
        cos_theta_m = torch.where(target_cos > self.th, cos_theta_m, target_cos - self.mm)

        logits = cos_theta.clone()
        logits[torch.arange(x.size(0)), labels] = cos_theta_m
        logits = logits * self.s

        return F.cross_entropy(logits, labels)

    def predict(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=1)
        return torch.argmax(F.linear(x_norm, W_norm), dim=1)

# ==========================================
# 2. TUNED Training Loop (Epochs & Lambda Warmup)
# ==========================================
def run_experiment(loss_type='decoupled', embed_size=2, epochs=20, batch_size=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n--- Running Pipeline: {loss_type.upper()} | Embed Size: {embed_size} | Dataset: MNIST ---")

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    dataset_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size, shuffle=False, num_workers=2)

    model = MNISTNet(embed_size=embed_size).to(device)
    num_classes = 10

    if loss_type == 'distarc':
        # Start with a very low lambda to allow angular learning first
        criterion = DistArcLoss(in_features=embed_size, out_features=num_classes, m=0.4, lam=0.0005).to(device)
    elif loss_type == 'decoupled':
        scale_s = 15.0 if embed_size == 2 else 30.0
        criterion = DecoupledLoss(in_features=embed_size, out_features=num_classes, s=scale_s).to(device)
    elif loss_type == 'arcface':
        # Now using the tuned 2D parameters
        criterion = ArcFaceLoss(in_features=embed_size, out_features=num_classes, s=15.0, m=0.25).to(device)
    elif loss_type == 'crossentropy':
        criterion = CrossEntropyLossWrapper(in_features=embed_size, out_features=num_classes).to(device)

    # Lower learning rate slightly for longer training stability
    optimizer = optim.SGD([
        {'params': model.parameters()},
        {'params': criterion.parameters()}
    ], lr=0.005, momentum=0.9, weight_decay=5e-4)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Training Loop
    for epoch in range(1, epochs + 1):

        # DISTARC LAMBDA WARMUP SCHEDULE (Paper Implementation)
        if loss_type == 'distarc':
            # Increase lambda by 0.0005 every 4 epochs until hitting max 0.003
            if epoch % 4 == 0 and criterion.lam < 0.003:
                criterion.lam += 0.0005
                print(f"   [Warmup] DistArc Lambda increased to: {criterion.lam:.4f}")

        model.train()
        criterion.train()
        running_loss = 0.0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()

            embeddings = model(data)
            loss = criterion(embeddings, target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    # Evaluation Loop
    model.eval()
    criterion.eval()
    correct = 0
    all_features = []
    all_labels = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            embeddings = model(data)

            preds = criterion.predict(embeddings)
            correct += preds.eq(target).sum().item()

            if embed_size == 2:
                all_features.append(embeddings.cpu().numpy())
                all_labels.append(target.cpu().numpy())

    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"Test Accuracy for {loss_type.upper()} (Dim={embed_size}): {accuracy:.2f}%")

    if embed_size == 2:
        features = np.vstack(all_features)
        labels = np.concatenate(all_labels)
        plot_title = f"{loss_type.capitalize()} Loss MNIST Latent Space (2D)"
        plot_filename = f"mnist_tuned_{loss_type}_2d_visualization.png"
        plot_features(features, labels, num_classes, plot_title, plot_filename)

    return accuracy
if __name__ == "__main__":
    # Test all 4 losses on MNIST with 2D Embeddings
    epochs = 20

    acc_ce = run_experiment(loss_type='crossentropy', embed_size=2, epochs=epochs)
    acc_arc = run_experiment(loss_type='arcface', embed_size=2, epochs=epochs)
    acc_distarc = run_experiment(loss_type='distarc', embed_size=2, epochs=epochs)
    acc_decoupled = run_experiment(loss_type='decoupled', embed_size=2, epochs=epochs)

    print("\n================ MNIST 2D SUMMARY ================")
    print(f"Cross Entropy Accuracy: {acc_ce:.2f}%")
    print(f"ArcFace Accuracy:       {acc_arc:.2f}%")
    print(f"DistArc Accuracy:       {acc_distarc:.2f}%")
    print(f"Decoupled Accuracy:     {acc_decoupled:.2f}%")


--- Running Pipeline: CROSSENTROPY | Embed Size: 2 | Dataset: MNIST ---
Epoch 1/20 - Loss: 1.0662
Epoch 2/20 - Loss: 0.3668
Epoch 3/20 - Loss: 0.2401
Epoch 4/20 - Loss: 0.1810
Epoch 5/20 - Loss: 0.1464
Epoch 6/20 - Loss: 0.1206
Epoch 7/20 - Loss: 0.1035
Epoch 8/20 - Loss: 0.0886
Epoch 9/20 - Loss: 0.0761
Epoch 10/20 - Loss: 0.0652
Epoch 11/20 - Loss: 0.0594
Epoch 12/20 - Loss: 0.0544
Epoch 13/20 - Loss: 0.0495
Epoch 14/20 - Loss: 0.0447
Epoch 15/20 - Loss: 0.0412
Epoch 16/20 - Loss: 0.0385
Epoch 17/20 - Loss: 0.0358
Epoch 18/20 - Loss: 0.0357
Epoch 19/20 - Loss: 0.0341
Epoch 20/20 - Loss: 0.0335
Test Accuracy for CROSSENTROPY (Dim=2): 97.73%
Saved visualization to mnist_tuned_crossentropy_2d_visualization.png

--- Running Pipeline: ARCFACE | Embed Size: 2 | Dataset: MNIST ---
Epoch 1/20 - Loss: 1.7695
Epoch 2/20 - Loss: 0.7804
Epoch 3/20 - Loss: 0.5172
Epoch 4/20 - Loss: 0.4144
Epoch 5/20 - Loss: 0.3742
Epoch 6/20 - Loss: 0.3307
Epoch 7/20 - Loss: 0.3052
Epoch 8/20 - Loss: 0.2763
Epoc